# NutriMatch All Predictions: Linear + Tree Models With Explainability

Archived Gen1 notebook for broad downstream prediction across paper-aligned NutriMatch targets.

It compares age/sex plus diet feature arms using:

- `linear`: Ridge regression for continuous targets and logistic regression for classification/quartile targets.
- `tree`: LightGBM with GPU when available and requested, otherwise sklearn HistGradientBoosting.

For continuous outcomes it also creates quartile-classification tasks when there are enough samples. Linear models save coefficient tables. Tree models try SHAP when the package supports the fitted backend, and skip gracefully when unavailable.

In [ ]:
from pathlib import Path
import gc
import json
import os
import re
import sys
import time
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import pearsonr
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    mean_squared_error,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, label_binarize

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

try:
    import shap
    SHAP_AVAILABLE = True
except Exception as exc:
    shap = None
    SHAP_AVAILABLE = False
    SHAP_IMPORT_ERROR = repr(exc)
else:
    SHAP_IMPORT_ERROR = None

try:
    from lightgbm import LGBMClassifier, LGBMRegressor
    LIGHTGBM_AVAILABLE = True
except Exception as exc:
    LGBMClassifier = None
    LGBMRegressor = None
    LIGHTGBM_AVAILABLE = False
    LIGHTGBM_IMPORT_ERROR = repr(exc)
else:
    LIGHTGBM_IMPORT_ERROR = None

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
pd.set_option('display.max_columns', 180)
pd.set_option('display.max_rows', 220)
sns.set_theme(style='whitegrid', context='notebook')

RUN_TRAINING = os.environ.get('DDE_RUN_TRAINING', '0') == '1'
RANDOM_STATE = int(os.environ.get('DDE_RANDOM_STATE', '42'))
N_SPLITS = int(os.environ.get('DDE_N_SPLITS', '5'))
MIN_N_PER_TARGET = int(os.environ.get('DDE_MIN_N_PER_TARGET', '80'))
MIN_CLASS_COUNT = int(os.environ.get('DDE_MIN_CLASS_COUNT', '20'))
MIN_DAILY_KCAL = float(os.environ.get('DDE_MIN_DAILY_KCAL', '800'))
X_BUILD_BATCH_SIZE = int(os.environ.get('DDE_X_BATCH_SIZE', '20'))
MAX_TARGETS = int(os.environ.get('DDE_MAX_TARGETS', '0'))  # 0 means no cap
EXPLAIN_TOP_N = int(os.environ.get('DDE_EXPLAIN_TOP_N', '40'))
SHAP_SAMPLE_N = int(os.environ.get('DDE_SHAP_SAMPLE_N', '500'))
SHAP_TOP_FEATURES = int(os.environ.get('DDE_SHAP_TOP_FEATURES', '50'))
USE_GPU = os.environ.get('DDE_USE_GPU', '1') == '1'
FEATURE_SET_FILTER = [x.strip() for x in os.environ.get('DDE_FEATURE_SET_FILTER', '').split(',') if x.strip()]
TARGET_ID_FILTER = [x.strip() for x in os.environ.get('DDE_TARGET_ID_FILTER', '').split(',') if x.strip()]
MODEL_FILTER = [x.strip() for x in os.environ.get('DDE_MODEL_FILTER', '').split(',') if x.strip()]
MODEL_FAMILIES = MODEL_FILTER or ['linear', 'tree']

print('RUN_TRAINING:', RUN_TRAINING)
print('Model families:', MODEL_FAMILIES)
print('LightGBM available:', LIGHTGBM_AVAILABLE, '| error:', LIGHTGBM_IMPORT_ERROR)
print('SHAP available:', SHAP_AVAILABLE, '| error:', SHAP_IMPORT_ERROR)

LIGHTGBM_GPU_AVAILABLE = False
LIGHTGBM_GPU_ERROR = None
if LIGHTGBM_AVAILABLE and USE_GPU:
    try:
        # Import-time availability is not enough; GPU LightGBM can fail only on fit.
        tiny_x = np.array([[0.0], [1.0], [2.0], [3.0]])
        tiny_y = np.array([0.0, 1.0, 0.0, 1.0])
        LGBMRegressor(n_estimators=2, device_type='gpu', verbose=-1, random_state=RANDOM_STATE).fit(tiny_x, tiny_y)
        LIGHTGBM_GPU_AVAILABLE = True
    except Exception as exc:
        LIGHTGBM_GPU_ERROR = repr(exc)

print('GPU requested:', USE_GPU)
print('LightGBM GPU usable:', LIGHTGBM_GPU_AVAILABLE, '| error:', LIGHTGBM_GPU_ERROR)
print('N_SPLITS:', N_SPLITS, '| MIN_N_PER_TARGET:', MIN_N_PER_TARGET)
print('MAX_TARGETS:', MAX_TARGETS or 'all')

In [ ]:
PROJECT_ROOT = Path.cwd()
tre_root = Path('/home/ec2-user/studies/Diet_Data_Enhancement_Project/Diet_Data_Enhancement_TRE')
if not (PROJECT_ROOT / 'downstream_analysis').exists() and (tre_root / 'downstream_analysis').exists():
    PROJECT_ROOT = tre_root
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

NOTEBOOK_STEM = 'nutrimatch_all_predictions_linear_hgb_explainability'
TASK_DIR = PROJECT_ROOT / 'depricated/Nutrimatch_manual/outputs' / NOTEBOOK_STEM
OUT_DIR = TASK_DIR / 'outputs'
FIG_DIR = OUT_DIR / 'figures'
CACHE_DIR = OUT_DIR / 'cache'
LOG_DIR = OUT_DIR / 'logs'
TRE_INPUTS = PROJECT_ROOT / 'tre_inputs'
SHARED_X_CACHE_DIRS = [
    PROJECT_ROOT / 'downstream_analysis/tasks/nutrimatch_two_year_obesity_with_enhancements/outputs/cache',
    PROJECT_ROOT / 'downstream_analysis/tasks/nutrimatch_baseline_fat_prediction_with_enhancements/outputs/cache',
    PROJECT_ROOT / 'depricated/Nutrimatch_manual/outputs/nutrimatch_two_year_obesity_with_enhancements/outputs/cache',
]
for d in [TASK_DIR, OUT_DIR, FIG_DIR, CACHE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

FEATURE_SETS = [
    {'name': 'basic_nutrimatch', 'label': 'NutriMatch all nutrients', 'path': 'outputs/enhanced_hpp/2.nutrimatch_based/hpp_feature_matrix_per_100g.csv', 'feature_mode': 'enriched'},
    {'name': 'denovo_microbiome', 'label': 'De novo microbiome-oriented', 'path': 'outputs/downstream_features/denovo/microbiome/hpp_downstream_feature_table.csv', 'feature_mode': 'kg'},
    {'name': 'denovo_cardiometabolic', 'label': 'De novo cardiometabolic', 'path': 'outputs/downstream_features/denovo/cardiometabolic/hpp_downstream_feature_table.csv', 'feature_mode': 'kg'},
    {'name': 'nutrimatch_microbiome', 'label': 'NutriMatch microbiome-oriented', 'path': 'outputs/downstream_features/nutrimatch_based/microbiome/hpp_downstream_feature_table.csv', 'feature_mode': 'kg'},
    {'name': 'nutrimatch_cardiometabolic', 'label': 'NutriMatch cardiometabolic', 'path': 'outputs/downstream_features/nutrimatch_based/cardiometabolic/hpp_downstream_feature_table.csv', 'feature_mode': 'kg'},
    {'name': 'nutrimatch_broad_diet_health', 'label': 'NutriMatch broad diet-health', 'path': 'outputs/downstream_features/nutrimatch_based/broad_diet_health/hpp_downstream_feature_table.csv', 'feature_mode': 'kg'},
    {'name': 'nutrimatch_mental_health', 'label': 'NutriMatch mental-health', 'path': 'outputs/downstream_features/nutrimatch_based/mental_health/hpp_downstream_feature_table.csv', 'feature_mode': 'kg'},
    {'name': 'denovo_broad_diet_health', 'label': 'De novo broad diet-health', 'path': 'outputs/downstream_features/denovo/broad_diet_health/hpp_downstream_feature_table.csv', 'feature_mode': 'kg'},
    {'name': 'denovo_mental_health', 'label': 'De novo mental-health', 'path': 'outputs/downstream_features/denovo/mental_health/hpp_downstream_feature_table.csv', 'feature_mode': 'kg'},
]
FEATURE_SETS = [fs for fs in FEATURE_SETS if (PROJECT_ROOT / fs['path']).exists()]
if FEATURE_SET_FILTER:
    wanted = set(FEATURE_SET_FILTER) | {'basic_nutrimatch'}
    FEATURE_SETS = [fs for fs in FEATURE_SETS if fs['name'] in wanted]

ARM_LABELS = {
    'age_sex_only': 'Age + sex',
    'paper_basic_nutrients': 'Age + sex + basic nutrients',
    'nutrimatch_all': 'Age + sex + NutriMatch all nutrients',
}
for fs in FEATURE_SETS:
    if fs['name'] != 'basic_nutrimatch':
        ARM_LABELS[fs['name']] = 'Age + sex + ' + fs['label']
ARM_ORDER = ['age_sex_only', 'paper_basic_nutrients', 'nutrimatch_all'] + [fs['name'] for fs in FEATURE_SETS if fs['name'] != 'basic_nutrimatch']

print('Project root:', PROJECT_ROOT)
print('Output directory:', OUT_DIR)
print('Feature sets found:', [fs['name'] for fs in FEATURE_SETS])
print('Shared X caches:', [str(p) for p in SHARED_X_CACHE_DIRS if p.exists()])

In [ ]:
from downstream_analysis.data_handelling.pheno_loader_export import (
    make_loader,
    load_table_from_loader,
    dataframe_with_index_columns,
)

def read_any(path):
    path = Path(path)
    if path.suffix.lower() == '.parquet':
        return pd.read_parquet(path)
    return pd.read_csv(path, low_memory=False)

def make_onehot():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)

def normalize_pid_series(s):
    return s.astype(str)

def clean_feature_name(x):
    return re.sub(r'[^a-z0-9]+', '_', str(x).strip().lower()).strip('_')

def find_participant_col(df):
    for col in ['participant_id', 'Participant_Study_ID', 'research_stage_id', 'user_id', 'RegistrationCode']:
        if col in df.columns:
            return col
    for col in df.columns:
        text = str(col).lower()
        if 'participant' in text or 'research_stage' in text:
            return col
    return None

def find_first_matching_column(df, patterns):
    compiled = [re.compile(p, re.IGNORECASE) for p in patterns]
    for col in df.columns:
        if any(p.search(str(col)) for p in compiled):
            return col
    return None

def try_load_pheno_table(dataset, table=None):
    try:
        loader = make_loader(dataset, age_sex_dataset=None, errors='warn')
        table_name = table or dataset
        try:
            df = load_table_from_loader(loader, dataset, table_name, required=False)
        except Exception:
            df = None
        if df is None:
            dfs = getattr(loader, 'dfs', {})
            if table_name in dfs:
                df = dataframe_with_index_columns(dfs[table_name])
            elif len(dfs) == 1:
                df = dataframe_with_index_columns(next(iter(dfs.values())))
        return df, loader
    except Exception as exc:
        print(f'Could not load {dataset}/{table or dataset}: {exc}')
        return None, None

PAPER_BASIC_NUTRIENT_NAMES = ['Energy', 'Protein', 'Total lipid (fat)', 'Carbohydrate, by difference', 'Fiber, total dietary', 'Sodium, Na', 'Water', 'Alcohol, ethyl']
BASIC_NUTRIENT_KEYS = {clean_feature_name(x) for x in PAPER_BASIC_NUTRIENT_NAMES}
PAPER_BASIC_PATTERNS = {
    'energy': re.compile(r'(^|_)energy($|_)|calorie|kcal', re.IGNORECASE),
    'protein': re.compile(r'(^|_)protein($|_)', re.IGNORECASE),
    'total_lipid_fat': re.compile(r'total_lipid|lipid|total_fat|(^|_)fat($|_)', re.IGNORECASE),
    'carbohydrate_by_difference': re.compile(r'carbohydrate|(^|_)carb($|_)', re.IGNORECASE),
    'fiber_total_dietary': re.compile(r'fiber|fibre', re.IGNORECASE),
    'sodium_na': re.compile(r'sodium|(^|_)na($|_)', re.IGNORECASE),
    'water': re.compile(r'(^|_)water($|_)', re.IGNORECASE),
    'alcohol_ethyl': re.compile(r'alcohol|ethyl', re.IGNORECASE),
}

def paper_basic_nutrient_kind(col):
    name = clean_feature_name(col)
    if name in BASIC_NUTRIENT_KEYS:
        return name
    for kind, pattern in PAPER_BASIC_PATTERNS.items():
        if pattern.search(name):
            return kind
    return None

In [ ]:
TARGET_TABLE_SPECS = [
    ('anthropometrics', 'anthropometrics'),
    ('body_composition', 'body_composition'),
    ('blood_tests', 'blood_tests'),
    ('cgm', 'cgm'),
    ('cgm', 'iglu'),
    ('cgm', 'iglu_daily'),
    ('nightingale_metabolomics', 'nightingale_metabolomics'),
]
COVARIATE_TABLE_SPECS = [
    ('anthropometrics', 'age_sex'),
    ('body_composition', 'age_sex'),
    ('blood_tests', 'age_sex'),
    ('cgm', 'age_sex'),
    ('nightingale_metabolomics', 'age_sex'),
    ('population', 'population'),
]
TARGET_REGEX = re.compile(
    r'body.*fat|body_comp.*fat|fat.*percent|fat_mass|visceral|vat|sat|'
    r'waist|hip|bmi|body_mass_index|'
    r'folate|folic|b9|glucose|glyca|hba1c|hemoglobin.*a1c|'
    r'cgm|time.*range|tir|mean.*glucose|average.*glucose|'
    r'gmi|j_index|mage|conga|modd|auc|cv_glucose|sd_glucose|'
    r'obes|overweight',
    re.IGNORECASE,
)
TARGET_CATEGORY_RULES = [
    ('anthropometry', re.compile(r'waist|hip|bmi|body_mass_index', re.IGNORECASE)),
    ('body_composition', re.compile(r'body_comp|body.*fat|fat.*percent|fat_mass|visceral|vat|sat', re.IGNORECASE)),
    ('blood_glucose', re.compile(r'glucose|glyca|hba1c|hemoglobin.*a1c', re.IGNORECASE)),
    ('blood_folate', re.compile(r'folate|folic|b9', re.IGNORECASE)),
    ('cgm', re.compile(r'cgm|time.*range|tir|mean.*glucose|average.*glucose|gmi|j_index|mage|conga|modd|auc|cv_glucose|sd_glucose', re.IGNORECASE)),
    ('obesity_status', re.compile(r'obes|overweight', re.IGNORECASE)),
]

def target_category(dataset, table, column):
    text = f'{dataset} {table} {column}'
    for category, pattern in TARGET_CATEGORY_RULES:
        if pattern.search(text):
            return category
    return 'other'

loaded_tables = {}
failed_table_specs = []
target_rows = []
dataset_briefs = []

for dataset, table in TARGET_TABLE_SPECS + COVARIATE_TABLE_SPECS:
    key = (dataset, table)
    if key in loaded_tables:
        continue
    frame, loader = try_load_pheno_table(dataset, table)
    if frame is None:
        failed_table_specs.append({'dataset': dataset, 'table': table})
        continue
    loaded_tables[key] = frame
    dataset_briefs.append({
        'dataset': dataset,
        'table': table,
        'rows': len(frame),
        'columns': len(frame.columns),
        'participant_col': find_participant_col(frame),
        'numeric_columns': len(frame.select_dtypes(include=np.number).columns),
    })

for (dataset, table), frame in loaded_tables.items():
    if (dataset, table) not in set(TARGET_TABLE_SPECS):
        continue
    pid_col = find_participant_col(frame)
    if pid_col is None:
        continue
    for col in frame.columns:
        if col == pid_col or not TARGET_REGEX.search(str(col)):
            continue
        s = frame[col]
        numeric = pd.api.types.is_numeric_dtype(s)
        nonnull = int(s.notna().sum())
        nunique = int(s.nunique(dropna=True))
        if nonnull == 0 or nunique < 2:
            continue
        if not numeric and nunique > 20:
            continue
        target_id = re.sub(r'[^A-Za-z0-9_]+', '_', f'{dataset}__{table}__{col}').strip('_')
        target_rows.append({
            'target_id': target_id,
            'category': target_category(dataset, table, col),
            'dataset': dataset,
            'table': table,
            'participant_col': pid_col,
            'column': col,
            'numeric': bool(numeric),
            'nonnull': nonnull,
            'nunique': nunique,
        })

dataset_briefs = pd.DataFrame(dataset_briefs).drop_duplicates()
target_catalog = pd.DataFrame(target_rows).drop_duplicates('target_id') if target_rows else pd.DataFrame()
failed_table_specs = pd.DataFrame(failed_table_specs)

dataset_briefs.to_csv(OUT_DIR / 'target_dataset_briefs.csv', index=False)
target_catalog.to_csv(OUT_DIR / 'candidate_target_catalog.csv', index=False)
failed_table_specs.to_csv(OUT_DIR / 'failed_target_table_loads.csv', index=False)
print('Loaded target/covariate tables:', len(loaded_tables))
print('Candidate targets:', len(target_catalog))
display(dataset_briefs)
display(target_catalog.sort_values(['category', 'dataset', 'column']).head(160) if not target_catalog.empty else target_catalog)

In [ ]:
PREFERRED_TARGET_TERMS = [
    'body_comp_total_region_percent_fat', 'total_scan_vat_volume', 'total_scan_vat_mass', 'total_scan_vat_area',
    'body_comp_trunk_region_percent_fat', 'waist_circumference', 'waist_to_hip_ratio', 'bmi',
    'folate', 'folic', 'bt__glucose_float_value', 'bt__hba1c_float_value', 'glyca',
    'mean_glucose', 'average_glucose', 'time_in_range', 'gmi', 'cgm_daily_above_250', 'obes', 'overweight',
]

def target_priority(row):
    text = f"{row['dataset']} {row['table']} {row['column']}".lower()
    for i, term in enumerate(PREFERRED_TARGET_TERMS):
        if term.lower() in text:
            return i
    category_order = {
        'body_composition': 100,
        'anthropometry': 200,
        'blood_folate': 300,
        'blood_glucose': 350,
        'cgm': 400,
        'obesity_status': 500,
        'other': 999,
    }
    return category_order.get(row.get('category', 'other'), 999)

if target_catalog.empty:
    raise ValueError('No candidate targets found. Check PhenoLoader tables and TARGET_REGEX.')

if TARGET_ID_FILTER:
    selected_catalog = target_catalog[target_catalog['target_id'].isin(TARGET_ID_FILTER)].copy()
    missing = sorted(set(TARGET_ID_FILTER) - set(selected_catalog['target_id']))
    if missing:
        print('Requested target IDs not found:', missing)
else:
    selected_catalog = target_catalog.copy()
    selected_catalog['priority'] = selected_catalog.apply(target_priority, axis=1)
    selected_catalog = selected_catalog.sort_values(['priority', 'nonnull'], ascending=[True, False])
    if MAX_TARGETS > 0:
        selected_catalog = selected_catalog.head(MAX_TARGETS)

selected_catalog.to_csv(OUT_DIR / 'selected_targets.csv', index=False)
print('Selected targets:', len(selected_catalog))
display(selected_catalog.head(160))

In [ ]:
def choose_participant_target_value(frame, pid_col, col):
    keep = [pid_col, col]
    date_cols = [c for c in ['collection_date', 'collection_timestamp', 'date', 'timestamp'] if c in frame.columns]
    part = frame[keep + date_cols].copy().rename(columns={pid_col: 'participant_id', col: 'target_value'})
    part['participant_id'] = normalize_pid_series(part['participant_id'])
    if pd.api.types.is_numeric_dtype(part['target_value']):
        part['target_value'] = pd.to_numeric(part['target_value'], errors='coerce')
    else:
        text = part['target_value'].astype(str).str.strip().str.lower()
        truthy = text.isin(['1', 'true', 'yes', 'y', 'case', 'positive', 'overweight', 'obese', 'obesity'])
        falsy = text.isin(['0', 'false', 'no', 'n', 'control', 'negative', 'normal', 'healthy'])
        part['target_value'] = np.where(truthy, 1.0, np.where(falsy, 0.0, np.nan))
    part = part.dropna(subset=['target_value'])
    if part.empty:
        return pd.DataFrame(columns=['participant_id', 'target_value'])
    if date_cols:
        date_col = date_cols[0]
        part[date_col] = pd.to_datetime(part[date_col], errors='coerce')
        part = part.sort_values(['participant_id', date_col]).groupby('participant_id', as_index=False).first()
    else:
        part = part.groupby('participant_id', as_index=False)['target_value'].mean()
    return part[['participant_id', 'target_value']]

target_parts = []
for _, row in selected_catalog.iterrows():
    frame = loaded_tables[(row['dataset'], row['table'])]
    part = choose_participant_target_value(frame, row['participant_col'], row['column']).rename(columns={'target_value': row['target_id']})
    target_parts.append(part)

targets_wide = target_parts[0]
for part in target_parts[1:]:
    targets_wide = targets_wide.merge(part, on='participant_id', how='outer')
targets_wide.to_csv(OUT_DIR / 'targets_participant_wide.csv', index=False)
print('Wrote targets:', OUT_DIR / 'targets_participant_wide.csv', targets_wide.shape)
display(targets_wide.head())

# Build executable task catalog: native regression/classification plus quartile classifications for regression targets.
task_rows = []
for _, row in selected_catalog.iterrows():
    target_id = row['target_id']
    y = pd.to_numeric(targets_wide[target_id], errors='coerce')
    y_non = y.dropna()
    if len(y_non) < MIN_N_PER_TARGET or y_non.nunique() < 2:
        continue
    is_binary = y_non.nunique() <= 2 or (not bool(row['numeric']))
    if is_binary:
        vc = y_non.astype(int).value_counts()
        if len(vc) >= 2 and vc.min() >= MIN_CLASS_COUNT:
            task_rows.append({**row.to_dict(), 'task_id': target_id + '__binary', 'task_type': 'classification', 'target_transform': 'binary', 'metric_primary': 'auprc'})
    else:
        task_rows.append({**row.to_dict(), 'task_id': target_id + '__regression', 'task_type': 'regression', 'target_transform': 'continuous', 'metric_primary': 'r2'})
        try:
            q = pd.qcut(y_non, 4, labels=False, duplicates='drop')
            if q.nunique() >= 3 and q.value_counts().min() >= MIN_CLASS_COUNT:
                task_rows.append({**row.to_dict(), 'task_id': target_id + '__quartile', 'task_type': 'classification', 'target_transform': 'quartile', 'metric_primary': 'f1_macro'})
        except Exception as exc:
            print('Quartile task skipped for', target_id, ':', exc)

task_catalog = pd.DataFrame(task_rows)
task_catalog.to_csv(OUT_DIR / 'task_catalog.csv', index=False)
print('Executable tasks:', len(task_catalog))
display(task_catalog[['task_id', 'target_id', 'category', 'task_type', 'target_transform', 'nonnull', 'nunique']].head(200))

In [ ]:
def load_covariates(participant_ids):
    cache = CACHE_DIR / 'covariates.csv'
    if cache.exists():
        cov = pd.read_csv(cache, low_memory=False)
        cov['participant_id'] = cov['participant_id'].astype(str)
        print('Loaded cached covariates:', cache, cov.shape)
        return cov
    frames = []
    for dataset, table in COVARIATE_TABLE_SPECS:
        frame = loaded_tables.get((dataset, table))
        if frame is None:
            continue
        pid_col = find_participant_col(frame)
        if pid_col is None:
            continue
        matches = [c for c in frame.columns if re.search(r'^age$|age_at|sex$|gender$|year_of_birth', str(c), re.IGNORECASE)]
        if not matches:
            continue
        part = frame[[pid_col] + matches].copy().rename(columns={pid_col: 'participant_id'})
        part['participant_id'] = normalize_pid_series(part['participant_id'])
        rename = {}
        for c in matches:
            lc = str(c).lower()
            if 'sex' in lc or 'gender' in lc:
                rename[c] = 'sex'
            elif 'year_of_birth' in lc:
                rename[c] = 'year_of_birth'
            elif 'age' in lc:
                rename[c] = 'age'
        part = part.rename(columns=rename)
        keep = ['participant_id'] + [c for c in ['age', 'sex', 'year_of_birth'] if c in part.columns]
        part = part[keep]
        if 'age' in part.columns:
            part['age'] = pd.to_numeric(part['age'], errors='coerce')
        if 'year_of_birth' in part.columns and 'age' not in part.columns:
            part['year_of_birth'] = pd.to_numeric(part['year_of_birth'], errors='coerce')
            part['age'] = 2022 - part['year_of_birth']
            part = part.drop(columns=['year_of_birth'])
        elif 'year_of_birth' in part.columns:
            part = part.drop(columns=['year_of_birth'])
        frames.append(part.groupby('participant_id', as_index=False).first())
    cov = pd.DataFrame({'participant_id': pd.Series(participant_ids).astype(str).unique()})
    for part in frames:
        for col in [c for c in part.columns if c != 'participant_id']:
            if col not in cov.columns:
                cov = cov.merge(part[['participant_id', col]], on='participant_id', how='left')
            else:
                add = part[['participant_id', col]].rename(columns={col: f'{col}_new'})
                cov = cov.merge(add, on='participant_id', how='left')
                cov[col] = cov[col].combine_first(cov[f'{col}_new'])
                cov = cov.drop(columns=[f'{col}_new'])
    if 'sex' in cov.columns:
        sex_text = cov['sex'].astype(str).str.lower()
        cov['sex'] = np.select(
            [sex_text.str.startswith('m') | sex_text.isin(['1', 'male']), sex_text.str.startswith('f') | sex_text.isin(['0', '2', 'female'])],
            ['male', 'female'],
            default=np.nan,
        )
    cov.to_csv(cache, index=False)
    print('Wrote covariates:', cache, cov.shape)
    return cov

covariates = load_covariates(targets_wide['participant_id'])
print('Covariate non-null counts:')
display(covariates.notna().sum())
display(covariates.head())

In [ ]:
def load_diet_events():
    for path in [TRE_INPUTS / 'diet_logging_events.parquet', TRE_INPUTS / 'diet_logging_events.csv']:
        if path.exists():
            print('Loading diet events:', path)
            return read_any(path)
    frame, _ = try_load_pheno_table('diet_logging', 'diet_logging_events')
    if frame is None:
        raise FileNotFoundError('Could not load diet_logging_events.')
    TRE_INPUTS.mkdir(parents=True, exist_ok=True)
    out = TRE_INPUTS / 'diet_logging_events.csv'
    frame.to_csv(out, index=False)
    return frame

def choose_day_col(df):
    if 'logging_day' in df.columns:
        return 'logging_day'
    for c in ['collection_date', 'local_date', 'date']:
        if c in df.columns:
            return c
    for c in ['collection_timestamp', 'local_timestamp', 'timestamp']:
        if c in df.columns:
            return c
    return None

def build_filtered_participant_food():
    own_cache = CACHE_DIR / f'diet_participant_food_gef_{int(MIN_DAILY_KCAL)}kcal.csv'
    own_days = CACHE_DIR / f'diet_valid_days_gef_{int(MIN_DAILY_KCAL)}kcal.csv'
    candidates = [(own_cache, own_days)]
    for d in SHARED_X_CACHE_DIRS:
        candidates.append((d / own_cache.name, d / own_days.name))
    for cache, days_cache in candidates:
        if cache.exists() and days_cache.exists():
            diet = pd.read_csv(cache, low_memory=False)
            valid_days = pd.read_csv(days_cache, low_memory=False)
            diet['participant_id'] = normalize_pid_series(diet['participant_id'])
            valid_days['participant_id'] = normalize_pid_series(valid_days['participant_id'])
            print('Loaded cached filtered diet:', cache, diet.shape)
            return diet, valid_days
    events = load_diet_events()
    required = ['participant_id', 'food_id', 'weight_g']
    missing = [c for c in required if c not in events.columns]
    if missing:
        raise ValueError(f'Diet events missing required columns: {missing}')
    day_col = choose_day_col(events)
    events['_diet_day'] = 'all_days' if day_col is None else events[day_col]
    if day_col and day_col != 'logging_day':
        events['_diet_day'] = pd.to_datetime(events['_diet_day'], errors='coerce').dt.date.astype(str)
    events['participant_id'] = normalize_pid_series(events['participant_id'])
    events['food_id'] = events['food_id'].astype(str)
    events['weight_g'] = pd.to_numeric(events['weight_g'], errors='coerce').fillna(0.0)
    kcal_col = find_first_matching_column(events, [r'calories_kcal', r'energy_kcal', r'kcal', r'calorie'])
    if kcal_col is not None:
        events['_kcal'] = pd.to_numeric(events[kcal_col], errors='coerce').fillna(0.0)
        day_energy = events.groupby(['participant_id', '_diet_day'], as_index=False)['_kcal'].sum()
        valid_day_keys = day_energy[day_energy['_kcal'] >= MIN_DAILY_KCAL][['participant_id', '_diet_day']]
        events = events.merge(valid_day_keys, on=['participant_id', '_diet_day'], how='inner')
        print('Applied kcal/day filter using', kcal_col, '| valid days:', len(valid_day_keys), 'of', len(day_energy))
    else:
        print('No calorie column found; aggregating all diet rows without kcal filter.')
    valid_days = events.groupby('participant_id', as_index=False)['_diet_day'].nunique().rename(columns={'_diet_day': 'valid_diet_days'})
    diet = events.groupby(['participant_id', 'food_id'], as_index=False)['weight_g'].sum()
    diet = diet.merge(valid_days, on='participant_id', how='left')
    diet.to_csv(own_cache, index=False)
    valid_days.to_csv(own_days, index=False)
    print('Wrote filtered diet:', own_cache, diet.shape)
    return diet, valid_days

def choose_ref_food_col(ref):
    for col in ['hpp_food_id', 'food_id']:
        if col in ref.columns:
            return col
    raise ValueError('Could not find hpp_food_id or food_id in feature table')

def feature_columns(ref, ref_food_col, feature_mode):
    if feature_mode == 'embedding':
        cols = [c for c in ref.columns if str(c).startswith('embedding_')]
    else:
        exclude = {ref_food_col, 'food_id', 'hpp_food_id', 'food_name', 'short_food_name', 'product_name'}
        cols = [c for c in ref.columns if c not in exclude and pd.api.types.is_numeric_dtype(ref[c])]
    return list(dict.fromkeys(cols))

def x_cache_candidates(fs_name):
    fname = f'X_{fs_name}_participant_gef_{int(MIN_DAILY_KCAL)}kcal.csv'
    return [CACHE_DIR / fname] + [d / fname for d in SHARED_X_CACHE_DIRS]

def build_participant_x(fs, batch_size=None):
    if batch_size is None:
        batch_size = X_BUILD_BATCH_SIZE
    for path in x_cache_candidates(fs['name']):
        if path.exists():
            x = pd.read_csv(path, low_memory=False)
            x['participant_id'] = normalize_pid_series(x['participant_id'])
            print('Loaded cached X:', path, x.shape)
            return x
    print('Building X:', fs['name'], '| batch_size=', batch_size, flush=True)
    ref = read_any(PROJECT_ROOT / fs['path'])
    ref_food_col = choose_ref_food_col(ref)
    cols = feature_columns(ref, ref_food_col, fs['feature_mode'])
    if not cols:
        raise ValueError(f"No numeric feature columns found for {fs['name']}")
    ref = ref[[ref_food_col] + cols].copy()
    ref['_food_join_id'] = ref[ref_food_col].astype(str)
    diet = participant_food.copy()
    diet['_food_join_id'] = diet['food_id'].astype(str)
    total_grams = diet.groupby('participant_id')['weight_g'].sum().replace(0, np.nan)
    days = valid_days.set_index('participant_id')['valid_diet_days'].replace(0, np.nan)
    parts = []
    for start in range(0, len(cols), batch_size):
        batch = cols[start:start + batch_size]
        print('  feature batch', start + 1, '-', min(start + batch_size, len(cols)), 'of', len(cols), flush=True)
        merged = diet[['participant_id', '_food_join_id', 'weight_g']].merge(ref[['_food_join_id'] + batch], on='_food_join_id', how='left')
        values = merged[batch].apply(pd.to_numeric, errors='coerce').fillna(0.0)
        if fs['feature_mode'] == 'enriched':
            scaled = values.mul(merged['weight_g'].to_numpy() / 100.0, axis=0)
            agg = scaled.assign(participant_id=merged['participant_id'].values).groupby('participant_id').sum(numeric_only=True)
            agg = agg.div(days, axis=0).fillna(0.0)
            prefix = 'enriched_daily_'
        elif fs['feature_mode'] in ['kg', 'embedding']:
            scaled = values.mul(merged['weight_g'].to_numpy(), axis=0)
            agg = scaled.assign(participant_id=merged['participant_id'].values).groupby('participant_id').sum(numeric_only=True)
            agg = agg.div(total_grams, axis=0).fillna(0.0)
            prefix = 'kg_weighted_' if fs['feature_mode'] == 'kg' else 'food_card_weighted_'
        else:
            raise ValueError(fs['feature_mode'])
        agg.columns = [prefix + str(c) for c in agg.columns]
        parts.append(agg)
        del merged, values, scaled, agg
        gc.collect()
    x = pd.concat(parts, axis=1).reset_index()
    out = CACHE_DIR / f"X_{fs['name']}_participant_gef_{int(MIN_DAILY_KCAL)}kcal.csv"
    x.to_csv(out, index=False)
    print('Wrote X:', out, x.shape, flush=True)
    return x

participant_food, valid_days = build_filtered_participant_food()
display(valid_days['valid_diet_days'].describe())

x_tables = {}
for fs in FEATURE_SETS:
    x_tables[fs['name']] = build_participant_x(fs)
    gc.collect()
print('Built/loaded X tables:', {k: v.shape for k, v in x_tables.items()})

In [ ]:
basic_x = x_tables.get('basic_nutrimatch')
if basic_x is None:
    raise ValueError('basic_nutrimatch feature table is required for the basic nutrient baseline.')
all_basic_feature_cols = [c for c in basic_x.columns if c != 'participant_id' and pd.api.types.is_numeric_dtype(basic_x[c])]
paper_basic_cols_by_kind = {}
for col in all_basic_feature_cols:
    kind = paper_basic_nutrient_kind(col)
    if kind and kind not in paper_basic_cols_by_kind:
        paper_basic_cols_by_kind[kind] = col
paper_basic_cols = list(paper_basic_cols_by_kind.values())
print('Paper-basic nutrient kinds found:', sorted(paper_basic_cols_by_kind))
print('Paper-basic nutrient columns found:', paper_basic_cols)
if not paper_basic_cols:
    raise ValueError('No paper-basic nutrient columns were found in basic_nutrimatch X.')

covariate_cols = [c for c in ['age', 'sex'] if c in covariates.columns and covariates[c].notna().any()]
print('Using covariates:', covariate_cols)

arms = [
    {'arm': 'age_sex_only', 'feature_set': 'age_sex_only', 'label': ARM_LABELS['age_sex_only'], 'x': covariates[['participant_id'] + covariate_cols].copy()},
    {'arm': 'paper_basic_nutrients', 'feature_set': 'paper_basic_nutrients', 'label': ARM_LABELS['paper_basic_nutrients'], 'x': basic_x[['participant_id'] + paper_basic_cols].merge(covariates[['participant_id'] + covariate_cols], on='participant_id', how='left')},
    {'arm': 'nutrimatch_all', 'feature_set': 'basic_nutrimatch', 'label': ARM_LABELS['nutrimatch_all'], 'x': basic_x.merge(covariates[['participant_id'] + covariate_cols], on='participant_id', how='left')},
]
for fs in FEATURE_SETS:
    if fs['name'] == 'basic_nutrimatch':
        continue
    x = x_tables[fs['name']].merge(covariates[['participant_id'] + covariate_cols], on='participant_id', how='left')
    arms.append({'arm': fs['name'], 'feature_set': fs['name'], 'label': ARM_LABELS[fs['name']], 'x': x})

arms = [a for a in arms if a['arm'] in ARM_ORDER]
for arm in arms:
    print(arm['arm'], arm['x'].shape, arm['label'])

In [ ]:
def transform_target(y_raw, task):
    y = pd.to_numeric(y_raw, errors='coerce')
    if task['target_transform'] == 'continuous':
        return y
    if task['target_transform'] == 'binary':
        return y.round().astype('Int64')
    if task['target_transform'] == 'quartile':
        out = pd.Series(pd.NA, index=y.index, dtype='Int64')
        non = y.dropna()
        q = pd.qcut(non, 4, labels=False, duplicates='drop')
        out.loc[q.index] = q.astype(int)
        return out
    raise ValueError(task['target_transform'])

def tree_backend_for(task_type):
    # sklearn HistGradientBoosting has no GPU path. If LightGBM is installed and GPU works,
    # the tree family will use it; otherwise it falls back to HistGradientBoosting.
    if LIGHTGBM_AVAILABLE and USE_GPU:
        try:
            if task_type == 'classification':
                return LGBMClassifier(n_estimators=20, learning_rate=0.05, device_type='gpu', random_state=RANDOM_STATE, verbose=-1)
            return LGBMRegressor(n_estimators=20, learning_rate=0.05, device_type='gpu', random_state=RANDOM_STATE, verbose=-1)
        except Exception:
            pass
    return None

def model_for(task_type, model_family):
    if model_family == 'linear':
        if task_type == 'regression':
            return Ridge(alpha=1.0, random_state=RANDOM_STATE)
        return LogisticRegression(max_iter=2000, class_weight='balanced', solver='lbfgs', multi_class='auto', random_state=RANDOM_STATE)
    if model_family == 'tree':
        if LIGHTGBM_AVAILABLE and LIGHTGBM_GPU_AVAILABLE:
            kwargs = dict(n_estimators=400, learning_rate=0.03, random_state=RANDOM_STATE, verbose=-1, device_type='gpu')
            return LGBMClassifier(**kwargs) if task_type == 'classification' else LGBMRegressor(**kwargs)
        if LIGHTGBM_AVAILABLE and os.environ.get('DDE_USE_LIGHTGBM_CPU', '0') == '1':
            kwargs = dict(n_estimators=400, learning_rate=0.03, random_state=RANDOM_STATE, verbose=-1, n_jobs=-1)
            return LGBMClassifier(**kwargs) if task_type == 'classification' else LGBMRegressor(**kwargs)
        if task_type == 'classification':
            return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.03, random_state=RANDOM_STATE)
        return HistGradientBoostingRegressor(max_iter=300, learning_rate=0.03, random_state=RANDOM_STATE)
    raise ValueError(model_family)

def effective_model_name(task_type, model_family):
    if model_family == 'linear':
        return 'linear_logistic' if task_type == 'classification' else 'linear_ridge'
    if LIGHTGBM_AVAILABLE and LIGHTGBM_GPU_AVAILABLE:
        return 'lightgbm_gpu'
    if LIGHTGBM_AVAILABLE and os.environ.get('DDE_USE_LIGHTGBM_CPU', '0') == '1':
        return 'lightgbm_cpu'
    return 'hist_gradient_boosting'

def pipeline_for(X, task_type, model_family):
    numeric_cols = X.select_dtypes(include=[np.number, bool]).columns.tolist()
    cat_cols = [c for c in X.columns if c not in numeric_cols]
    if model_family == 'linear':
        num_steps = [('impute', SimpleImputer()), ('scale', StandardScaler())]
    else:
        num_steps = [('impute', SimpleImputer())]
    pre = ColumnTransformer(
        transformers=[
            ('num', Pipeline(num_steps), numeric_cols),
            ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('onehot', make_onehot())]), cat_cols),
        ],
        remainder='drop',
    )
    return Pipeline([('pre', pre), ('model', model_for(task_type, model_family))])

def feature_names_from_preprocessor(fitted_pipeline, original_X):
    pre = fitted_pipeline.named_steps['pre']
    try:
        return pre.get_feature_names_out().tolist()
    except Exception:
        names = []
        for name, transformer, cols in pre.transformers_:
            if name == 'remainder' or transformer == 'drop':
                continue
            if name == 'num':
                names.extend([f'num__{c}' for c in cols])
            elif name == 'cat':
                try:
                    onehot = transformer.named_steps['onehot']
                    names.extend(onehot.get_feature_names_out(cols).tolist())
                except Exception:
                    names.extend([f'cat__{c}' for c in cols])
        return names

def usable_cv(y, task_type):
    if task_type == 'classification':
        counts = y.value_counts(dropna=True)
        if len(counts) < 2 or counts.min() < 2:
            return None
        k = min(N_SPLITS, int(counts.min()))
        return StratifiedKFold(n_splits=k, shuffle=True, random_state=RANDOM_STATE) if k >= 2 else None
    k = min(N_SPLITS, int(y.notna().sum()))
    return KFold(n_splits=k, shuffle=True, random_state=RANDOM_STATE) if k >= 2 else None

def classification_scores(y_test, pred, proba, classes):
    row = {
        'accuracy': float(accuracy_score(y_test, pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_test, pred)),
        'f1_macro': float(f1_score(y_test, pred, average='macro', zero_division=0)),
        'auroc': np.nan,
        'auprc': np.nan,
    }
    try:
        if proba is not None and len(classes) == 2:
            score = proba[:, 1]
            positive = classes[1]
            row['auroc'] = float(roc_auc_score(y_test, score))
            row['auprc'] = float(average_precision_score((pd.Series(y_test).to_numpy() == positive).astype(int), score))
        elif proba is not None and len(classes) > 2:
            row['auroc'] = float(roc_auc_score(y_test, proba, multi_class='ovr', average='macro', labels=classes))
            y_bin = label_binarize(y_test, classes=classes)
            row['auprc'] = float(average_precision_score(y_bin, proba, average='macro'))
    except Exception:
        pass
    return row

def evaluate_one(arm, task, model_family):
    target_id = task['target_id']
    y_table = targets_wide[['participant_id', target_id]].copy()
    y_table['participant_id'] = normalize_pid_series(y_table['participant_id'])
    y_table['y'] = transform_target(y_table[target_id], task)
    merged = arm['x'].merge(y_table[['participant_id', 'y']], on='participant_id', how='inner')
    X = merged.drop(columns=['participant_id', 'y'])
    y = merged['y']
    keep = y.notna()
    X = X.loc[keep].copy()
    y = y.loc[keep].copy()
    if len(y) < MIN_N_PER_TARGET:
        return None, None, None
    if task['task_type'] == 'classification':
        y = y.astype(int).astype(str)
        counts = y.value_counts()
        if len(counts) < 2 or counts.min() < MIN_CLASS_COUNT:
            return None, None, None
    cv = usable_cv(y, task['task_type'])
    if cv is None:
        return None, None, None
    estimator = pipeline_for(X, task['task_type'], model_family)
    split_iter = cv.split(X, y) if task['task_type'] == 'classification' else cv.split(X)
    fold_rows, oof_rows = [], []
    for fold, (train_idx, test_idx) in enumerate(split_iter, start=1):
        est = clone(estimator)
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        est.fit(X_train, y_train)
        pred = est.predict(X_test)
        if task['task_type'] == 'classification':
            proba = est.predict_proba(X_test) if hasattr(est, 'predict_proba') else None
            classes = list(est.named_steps['model'].classes_) if hasattr(est.named_steps['model'], 'classes_') else sorted(y.unique())
            row = classification_scores(y_test, pred, proba, classes)
            if proba is not None:
                if len(classes) == 2:
                    score = proba[:, 1]
                else:
                    score = proba.max(axis=1)
            else:
                score = np.full(len(test_idx), np.nan)
            for idx, yt, yp, sc in zip(test_idx, y_test, pred, score):
                oof_rows.append({'row_index': int(idx), 'fold': fold, 'y_true': str(yt), 'y_pred': str(yp), 'y_score': float(sc) if pd.notna(sc) else np.nan})
        else:
            try:
                pr = float(pearsonr(pd.to_numeric(y_test), pd.to_numeric(pred)).statistic)
            except Exception:
                pr = np.nan
            row = {
                'r2': float(r2_score(y_test, pred)),
                'rmse': float(np.sqrt(mean_squared_error(y_test, pred))),
                'pearson_r': pr,
            }
            for idx, yt, yp in zip(test_idx, y_test, pred):
                oof_rows.append({'row_index': int(idx), 'fold': fold, 'y_true': float(yt), 'y_pred': float(yp), 'y_score': float(yp)})
        row['fold'] = fold
        row['n_test'] = int(len(test_idx))
        fold_rows.append(row)
    fold_df = pd.DataFrame(fold_rows)
    oof_df = pd.DataFrame(oof_rows)
    summary = {
        'task_id': task['task_id'],
        'target_id': task['target_id'],
        'target_column': task['column'],
        'target_dataset': task['dataset'],
        'target_table': task['table'],
        'category': task['category'],
        'task_type': task['task_type'],
        'target_transform': task['target_transform'],
        'arm': arm['arm'],
        'feature_set': arm['feature_set'],
        'label': arm['label'],
        'model_family': model_family,
        'model': effective_model_name(task['task_type'], model_family),
        'n': int(len(y)),
        'feature_count': int(X.shape[1]),
    }
    if task['task_type'] == 'classification':
        summary.update({
            'class_count': int(y.nunique()),
            'class_counts': json.dumps(y.value_counts().to_dict()),
        })
    for col in fold_df.columns:
        if col in ['fold', 'n_test']:
            continue
        summary[f'{col}_mean'] = float(fold_df[col].mean())
        summary[f'{col}_std'] = float(fold_df[col].std())
    primary = 'r2_mean' if task['task_type'] == 'regression' else ('auprc_mean' if pd.notna(summary.get('auprc_mean', np.nan)) else 'f1_macro_mean')
    summary['primary_metric_name'] = primary.replace('_mean', '')
    summary['primary_metric'] = summary.get(primary, np.nan)
    fold_df['task_id'] = task['task_id']; fold_df['arm'] = arm['arm']; fold_df['model_family'] = model_family; fold_df['model'] = summary['model']
    oof_df['task_id'] = task['task_id']; oof_df['arm'] = arm['arm']; oof_df['model_family'] = model_family; oof_df['model'] = summary['model']
    return summary, fold_df, oof_df

In [ ]:
results_path = OUT_DIR / 'all_prediction_model_results.csv'
fold_path = OUT_DIR / 'all_prediction_fold_metrics.csv'
oof_path = OUT_DIR / 'all_prediction_oof_predictions.csv'

if RUN_TRAINING:
    summaries, fold_tables, oof_tables = [], [], []
    for _, task in task_catalog.iterrows():
        print('\nTarget task:', task['task_id'], '|', task['task_type'], '|', task['target_transform'], flush=True)
        for arm in arms:
            for model_family in MODEL_FAMILIES:
                try:
                    summary, fold_df, oof_df = evaluate_one(arm, task, model_family)
                except Exception as exc:
                    print('  ERROR', arm['arm'], model_family, ':', repr(exc), flush=True)
                    continue
                if summary is None:
                    print('  skipped', arm['arm'], model_family, flush=True)
                    continue
                summaries.append(summary)
                fold_tables.append(fold_df)
                oof_tables.append(oof_df)
                print(' ', arm['arm'], model_family, summary['primary_metric_name'], summary['primary_metric'], flush=True)
                if len(summaries) % 20 == 0:
                    pd.DataFrame(summaries).to_csv(results_path, index=False)
    results = pd.DataFrame(summaries)
    fold_metrics = pd.concat(fold_tables, ignore_index=True) if fold_tables else pd.DataFrame()
    oof_predictions = pd.concat(oof_tables, ignore_index=True) if oof_tables else pd.DataFrame()
    results.to_csv(results_path, index=False)
    fold_metrics.to_csv(fold_path, index=False)
    oof_predictions.to_csv(oof_path, index=False)
    print('Wrote:', results_path)
    print('Wrote:', fold_path)
    print('Wrote:', oof_path)
elif results_path.exists():
    results = pd.read_csv(results_path, low_memory=False)
    fold_metrics = pd.read_csv(fold_path, low_memory=False) if fold_path.exists() else pd.DataFrame()
    oof_predictions = pd.read_csv(oof_path, low_memory=False) if oof_path.exists() else pd.DataFrame()
    print('Loaded saved results:', results_path, results.shape)
else:
    results = pd.DataFrame(); fold_metrics = pd.DataFrame(); oof_predictions = pd.DataFrame()
    print('Training skipped and no saved results found yet.')

display(results.head() if not results.empty else results)

In [ ]:
coef_path = OUT_DIR / 'linear_coefficients.csv'
shap_importance_path = OUT_DIR / 'tree_shap_feature_importance.csv'
shap_dep_path = OUT_DIR / 'tree_shap_dependence_long.csv'
explain_manifest_path = OUT_DIR / 'explainability_manifest.csv'

def fit_full_model_for_result(row):
    task = task_catalog[task_catalog['task_id'].eq(row['task_id'])].iloc[0]
    arm = next(a for a in arms if a['arm'] == row['arm'])
    y_table = targets_wide[['participant_id', task['target_id']]].copy()
    y_table['participant_id'] = normalize_pid_series(y_table['participant_id'])
    y_table['y'] = transform_target(y_table[task['target_id']], task)
    merged = arm['x'].merge(y_table[['participant_id', 'y']], on='participant_id', how='inner')
    X = merged.drop(columns=['participant_id', 'y'])
    y = merged['y']
    keep = y.notna()
    X = X.loc[keep].copy()
    y = y.loc[keep].copy()
    if task['task_type'] == 'classification':
        y = y.astype(int).astype(str)
    est = pipeline_for(X, task['task_type'], row['model_family'])
    est.fit(X, y)
    return est, X, y, task

def extract_linear_coefficients(est, X, row):
    names = feature_names_from_preprocessor(est, X)
    model = est.named_steps['model']
    coef = getattr(model, 'coef_', None)
    if coef is None:
        return pd.DataFrame()
    coef = np.asarray(coef)
    if coef.ndim == 1:
        coef = coef.reshape(1, -1)
    rows = []
    classes = getattr(model, 'classes_', [None] * coef.shape[0])
    for class_ix in range(coef.shape[0]):
        class_name = None if class_ix >= len(classes) else classes[class_ix]
        for feature, value in zip(names, coef[class_ix]):
            rows.append({**row.to_dict(), 'class_name': class_name, 'feature': feature, 'coefficient': float(value), 'abs_coefficient': float(abs(value))})
    return pd.DataFrame(rows)

def compute_tree_shap(est, X, row):
    if not SHAP_AVAILABLE:
        return pd.DataFrame(), pd.DataFrame(), 'SHAP not installed'
    try:
        if len(X) > SHAP_SAMPLE_N:
            X_sample = X.sample(SHAP_SAMPLE_N, random_state=RANDOM_STATE)
        else:
            X_sample = X.copy()
        Xt = est.named_steps['pre'].transform(X_sample)
        names = feature_names_from_preprocessor(est, X)
        model = est.named_steps['model']
        explainer = shap.TreeExplainer(model)
        values = explainer.shap_values(Xt)
        if isinstance(values, list):
            arr = np.mean([np.abs(np.asarray(v)) for v in values], axis=0)
            signed = np.asarray(values[-1])
        else:
            arr_raw = np.asarray(values)
            if arr_raw.ndim == 3:
                arr = np.mean(np.abs(arr_raw), axis=2)
                signed = arr_raw[:, :, -1]
            else:
                arr = np.abs(arr_raw)
                signed = arr_raw
        if arr.ndim == 1:
            mean_abs = arr
        else:
            mean_abs = arr.mean(axis=0)
        if len(names) != len(mean_abs):
            names = [f'feature_{i}' for i in range(len(mean_abs))]
        imp = pd.DataFrame({**{k: row[k] for k in row.index}, 'feature': names, 'mean_abs_shap': mean_abs}).sort_values('mean_abs_shap', ascending=False)
        top_features = imp.head(SHAP_TOP_FEATURES)['feature'].tolist()
        dep_rows = []
        Xt_df = pd.DataFrame(Xt, columns=names)
        signed_df = pd.DataFrame(signed, columns=names) if np.asarray(signed).ndim == 2 and np.asarray(signed).shape[1] == len(names) else None
        if signed_df is not None:
            for feature in top_features:
                for i, (fv, sv) in enumerate(zip(Xt_df[feature].to_numpy(), signed_df[feature].to_numpy())):
                    dep_rows.append({**{k: row[k] for k in row.index}, 'sample_index': int(i), 'feature': feature, 'feature_value': float(fv), 'shap_value': float(sv)})
        return imp, pd.DataFrame(dep_rows), None
    except Exception as exc:
        return pd.DataFrame(), pd.DataFrame(), repr(exc)

if RUN_TRAINING and not results.empty:
    explain_rows = results.dropna(subset=['primary_metric']).sort_values('primary_metric', ascending=False).head(EXPLAIN_TOP_N).copy()
    coef_tables, shap_imps, shap_deps, manifest = [], [], [], []
    for _, row in explain_rows.iterrows():
        print('Explain:', row['model_family'], row['task_id'], row['arm'], flush=True)
        try:
            est, X, y, task = fit_full_model_for_result(row)
            status = 'ok'
            note = ''
            if row['model_family'] == 'linear':
                coef_tables.append(extract_linear_coefficients(est, X, row))
            elif row['model_family'] == 'tree':
                imp, dep, err = compute_tree_shap(est, X, row)
                if err:
                    status = 'skipped'
                    note = err
                    print('  SHAP skipped:', err, flush=True)
                else:
                    shap_imps.append(imp)
                    shap_deps.append(dep)
            manifest.append({**row.to_dict(), 'explain_status': status, 'explain_note': note})
        except Exception as exc:
            manifest.append({**row.to_dict(), 'explain_status': 'error', 'explain_note': repr(exc)})
            print('  explain error:', repr(exc), flush=True)
    linear_coefficients = pd.concat(coef_tables, ignore_index=True) if coef_tables else pd.DataFrame()
    shap_importance = pd.concat(shap_imps, ignore_index=True) if shap_imps else pd.DataFrame()
    shap_dependence = pd.concat(shap_deps, ignore_index=True) if shap_deps else pd.DataFrame()
    explain_manifest = pd.DataFrame(manifest)
    linear_coefficients.to_csv(coef_path, index=False)
    shap_importance.to_csv(shap_importance_path, index=False)
    shap_dependence.to_csv(shap_dep_path, index=False)
    explain_manifest.to_csv(explain_manifest_path, index=False)
    print('Wrote explainability artifacts.')
else:
    linear_coefficients = pd.read_csv(coef_path, low_memory=False) if coef_path.exists() else pd.DataFrame()
    shap_importance = pd.read_csv(shap_importance_path, low_memory=False) if shap_importance_path.exists() else pd.DataFrame()
    shap_dependence = pd.read_csv(shap_dep_path, low_memory=False) if shap_dep_path.exists() else pd.DataFrame()
    explain_manifest = pd.read_csv(explain_manifest_path, low_memory=False) if explain_manifest_path.exists() else pd.DataFrame()
    print('Loaded/skipped explainability artifacts:', linear_coefficients.shape, shap_importance.shape, shap_dependence.shape)

display(explain_manifest.head() if 'explain_manifest' in globals() and not explain_manifest.empty else pd.DataFrame())

In [ ]:
def save_fig(path):
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=240, bbox_inches='tight')
    print('Wrote:', path)

def short_point_label(row):
    label = row.get('target_column') or row.get('target_id') or row.get('task_id')
    label = str(label)
    label = re.sub(r'^(body_composition__body_composition__|anthropometrics__anthropometrics__|blood_tests__blood_tests__|cgm__iglu_daily__|cgm__iglu__|cgm__cgm__)', '', label)
    label = label.replace('body_comp_', '').replace('_float_value', '').replace('__regression', '').replace('__quartile', '').replace('__binary', '')
    label = re.sub(r'^bt__', '', label)
    return label[:42]

def add_point_labels(ax, comp, metric, label_points=True):
    if not label_points:
        return
    # Each x-position is the basic-nutrient baseline for one target/task.
    # Label only the highest-performing comparison arm at that x-position.
    label_df = (
        comp.dropna(subset=['basic_metric', metric])
        .sort_values(metric, ascending=False)
        .groupby(['task_id', 'model_family'], as_index=False)
        .head(1)
    )
    texts = []
    for _, row in label_df.iterrows():
        text = ax.annotate(
            short_point_label(row),
            xy=(row['basic_metric'], row[metric]),
            xytext=(5, 5),
            textcoords='offset points',
            fontsize=7,
            fontweight='semibold',
            alpha=0.9,
        )
        texts.append(text)
    try:
        from adjustText import adjust_text
        adjust_text(
            texts,
            ax=ax,
            arrowprops=dict(arrowstyle='-', color='0.55', lw=0.4, alpha=0.6),
            expand_points=(1.1, 1.2),
            expand_text=(1.05, 1.15),
        )
    except Exception:
        pass

def compare_against_basic(metric, task_type, model_family=None, title=None, path=None, label_points=True):
    if results.empty:
        print('No results loaded.')
        return pd.DataFrame()
    df = results[results['task_type'].eq(task_type)].copy()
    if model_family:
        df = df[df['model_family'].eq(model_family)]
    if metric not in df.columns:
        print('Metric not available:', metric)
        return pd.DataFrame()
    base = df[df['arm'].eq('paper_basic_nutrients')][['task_id', 'model_family', metric]].rename(columns={metric: 'basic_metric'})
    comp = df[~df['arm'].isin(['age_sex_only', 'paper_basic_nutrients'])].merge(base, on=['task_id', 'model_family'], how='inner')
    comp = comp.dropna(subset=['basic_metric', metric])
    if comp.empty:
        print('No comparable rows for', metric, task_type, model_family)
        return comp
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.scatterplot(data=comp, x='basic_metric', y=metric, hue='label', style='category', s=70, alpha=0.85, ax=ax)
    add_point_labels(ax, comp, metric, label_points=label_points)
    lo = min(comp['basic_metric'].min(), comp[metric].min())
    hi = max(comp['basic_metric'].max(), comp[metric].max())
    pad = (hi - lo) * 0.08 if hi > lo else 0.05
    ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], color='black', ls=':', lw=1)
    ax.set_xlim(lo - pad, hi + pad)
    ax.set_ylim(lo - pad, hi + pad)
    ax.set_xlabel(f'Age + sex + basic nutrients ({metric})')
    ax.set_ylabel(f'Comparison arms ({metric})')
    ax.set_title(title or f'{metric}: basic nutrients vs enhanced/NutriMatch arms')
    ax.legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc='upper left')
    if path:
        save_fig(path)
    plt.show()
    return comp

for mf in sorted(results['model_family'].dropna().unique()) if not results.empty else []:
    compare_against_basic('r2_mean', 'regression', mf, f'Regression R2: basic nutrients vs all arms ({mf})', FIG_DIR / f'scatter_basic_vs_all_regression_r2_{mf}.png')
    compare_against_basic('auprc_mean', 'classification', mf, f'Classification AUPRC: basic nutrients vs all arms ({mf})', FIG_DIR / f'scatter_basic_vs_all_classification_auprc_{mf}.png')
    compare_against_basic('f1_macro_mean', 'classification', mf, f'Classification F1 macro: basic nutrients vs all arms ({mf})', FIG_DIR / f'scatter_basic_vs_all_classification_f1_macro_{mf}.png')

In [ ]:
def plot_all_shap_for_model(model='lightgbm_gpu', top_n=25):
    if shap_importance.empty:
        print('No SHAP importance table loaded. It may have been skipped because SHAP/backend support was unavailable.')
        return
    df = shap_importance[shap_importance['model'].eq(model)].copy()
    if df.empty:
        print('No SHAP rows for model:', model)
        print('Available models:', sorted(shap_importance['model'].dropna().unique()))
        return
    combos = df[['task_id', 'arm', 'label']].drop_duplicates().head(12)
    for _, combo in combos.iterrows():
        sub = df[(df['task_id'].eq(combo['task_id'])) & (df['arm'].eq(combo['arm']))].sort_values('mean_abs_shap', ascending=False).head(top_n)
        plt.figure(figsize=(7, max(4, 0.25 * len(sub))))
        sns.barplot(data=sub, x='mean_abs_shap', y='feature', color='#4C72B0')
        plt.title(f"{model}: {combo['task_id']} | {combo['label']}")
        plt.xlabel('Mean absolute SHAP')
        plt.ylabel('')
        plt.tight_layout()
        plt.show()

def plot_shap_dependence(model='lightgbm_gpu', feature=None, task_id=None, arm=None):
    if shap_dependence.empty:
        print('No SHAP dependence table loaded.')
        return
    df = shap_dependence[shap_dependence['model'].eq(model)].copy()
    if task_id is not None:
        df = df[df['task_id'].eq(task_id)]
    if arm is not None:
        df = df[df['arm'].eq(arm)]
    if feature is None:
        print('Choose one feature from:')
        display(pd.Series(sorted(df['feature'].dropna().unique())).head(100).to_frame('feature'))
        return
    df = df[df['feature'].eq(feature)]
    if df.empty:
        print('No rows for that model/task/arm/feature combination.')
        return
    plt.figure(figsize=(7, 5))
    sns.scatterplot(data=df, x='feature_value', y='shap_value', hue='label', alpha=0.55, s=25)
    plt.axhline(0, color='black', lw=0.8)
    plt.title(f'SHAP dependence: {feature}')
    plt.xlabel('Transformed feature value')
    plt.ylabel('SHAP value')
    plt.legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

try:
    import ipywidgets as widgets
    from IPython.display import clear_output
    if not shap_importance.empty:
        model_dd = widgets.Dropdown(options=sorted(shap_importance['model'].dropna().unique()), description='Model')
        out = widgets.Output()
        def _show_all(change=None):
            with out:
                clear_output(wait=True)
                plot_all_shap_for_model(model_dd.value)
        model_dd.observe(_show_all, names='value')
        display(widgets.VBox([model_dd, out]))
        _show_all()
    if not shap_dependence.empty:
        model2 = widgets.Dropdown(options=sorted(shap_dependence['model'].dropna().unique()), description='Model')
        task2 = widgets.Dropdown(options=[''] + sorted(shap_dependence['task_id'].dropna().unique()), description='Task')
        arm2 = widgets.Dropdown(options=[''] + sorted(shap_dependence['arm'].dropna().unique()), description='Arm')
        feat2 = widgets.Combobox(options=sorted(shap_dependence['feature'].dropna().unique()), description='Feature', ensure_option=True)
        out2 = widgets.Output()
        def _show_dep(change=None):
            with out2:
                clear_output(wait=True)
                plot_shap_dependence(model2.value, feat2.value or None, task2.value or None, arm2.value or None)
        for w in [model2, task2, arm2, feat2]:
            w.observe(_show_dep, names='value')
        display(widgets.VBox([widgets.HBox([model2, task2, arm2]), feat2, out2]))
        if feat2.options:
            feat2.value = feat2.options[0]
            _show_dep()
except Exception as exc:
    print('Interactive widgets unavailable; use plot_all_shap_for_model(...) and plot_shap_dependence(...). Error:', repr(exc))

In [ ]:
if not results.empty:
    summary_cols = ['task_type', 'target_transform', 'model_family', 'model', 'arm', 'label', 'n', 'feature_count', 'r2_mean', 'auprc_mean', 'f1_macro_mean', 'primary_metric']
    existing = [c for c in summary_cols if c in results.columns]
    arm_summary = results.groupby(['task_type', 'target_transform', 'model_family', 'model', 'arm', 'label'], as_index=False).agg(
        tasks=('task_id', 'nunique'),
        mean_primary_metric=('primary_metric', 'mean'),
        mean_r2=('r2_mean', 'mean'),
        mean_auprc=('auprc_mean', 'mean'),
        mean_f1_macro=('f1_macro_mean', 'mean'),
    )
    arm_summary.to_csv(OUT_DIR / 'arm_summary.csv', index=False)
    paper_ready = results[existing].sort_values(['task_type', 'model_family', 'primary_metric'], ascending=[True, True, False])
    paper_ready.to_csv(OUT_DIR / 'paper_ready_all_prediction_results.csv', index=False)
    print('Wrote:', OUT_DIR / 'arm_summary.csv')
    print('Wrote:', OUT_DIR / 'paper_ready_all_prediction_results.csv')
    display(arm_summary.sort_values(['task_type', 'model_family', 'mean_primary_metric'], ascending=[True, True, False]).head(100))
else:
    print('No result table loaded yet.')